In [ ]:
import tensorflow as tf
import os
import numpy as np
# from osgeo import gdal, osr
# import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
import torch
import torch.nn as nn
from transformers import SegformerForSemanticSegmentation

class SegFormerSegmentor(nn.Module):
    def __init__(self, num_classes, num_channels=4):
        super(SegFormerSegmentor, self).__init__()
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0",
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        # Adjust input conv layer for different channel numbers (default is 3 for RGB)
        if num_channels != 3:
            old_conv = self.model.segformer.encoder.patch_embeddings[0].proj
            new_conv = nn.Conv2d(num_channels,
                                 old_conv.out_channels,
                                 kernel_size=old_conv.kernel_size,
                                 stride=old_conv.stride,
                                 padding=old_conv.padding,
                                 bias=old_conv.bias is not None)
            # Copy weights for first 3 channels if available, random init for the rest
            with torch.no_grad():
                new_conv.weight[:, :3, :, :] = old_conv.weight
                if num_channels > 3:
                    nn.init.xavier_uniform_(new_conv.weight[:, 3:, :, :])
            self.model.segformer.encoder.patch_embeddings[0].proj = new_conv

    def forward(self, x):
        """
        x: torch.Tensor with shape [B, C, H, W]
        """
        outputs = self.model(x)
        logits = outputs.logits  # [B, num_classes, H/4, W/4] (usually smaller resolution)
        # Upsample back to input resolution
        logits = nn.functional.interpolate(
            logits,
            size=(x.shape[2], x.shape[3]),
            mode="bilinear",
            align_corners=False
        )
        return logits


# Example usage
if __name__ == "__main__":
    model = SegFormerSegmentor(num_classes=10, num_channels=4)
    dummy_input = torch.randn(2, 4, 1024, 1024)  # [batch, channels, height, width]
    output = model(dummy_input)
    print(output.shape)  # torch.Size([2, 10, 256, 256])

/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch

torch.Size([2, 10, 1024, 1024])


In [4]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1200 #Vermontlc
num_sample = 2000
num_training = 1600
class_num=9
num_test = num_sample-num_training
dataset_name = 'VermontLC'
start_class = 0

In [5]:
file_folder = '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()
output_tfrecords_files

['/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_ATD_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_BiDiff_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_CAMixer_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_CFAT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_ESRGAN_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_RGT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_SED_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_SRNO_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_UPSR_x16.tfrecords']

In [6]:
file_folder = '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()

for output_tfrecords in output_tfrecords_files:
    dirname = '/glade/derecho/scratch/lizhili/s2naip/segformer_pths_16x/'  # one level up (/content/drive/MyDrive/GeoSR)
    filename = os.path.basename(output_tfrecords)                 # e.g. M2L8_River_SRNO_x4.tfrecords
    base = filename.replace(f"{dataset_name}_", "").replace("_x16.tfrecords", "")
    segformer_save_path =  os.path.join(dirname, f"{dataset_name}_Segformer_{base}_run2.pth")
    print(output_tfrecords, " Save to:", segformer_save_path)

/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_ATD_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/segformer_pths_16x/VermontLC_Segformer_ATD_run2.pth
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_BiDiff_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/segformer_pths_16x/VermontLC_Segformer_BiDiff_run2.pth
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_CAMixer_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/segformer_pths_16x/VermontLC_Segformer_CAMixer_run2.pth
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_CFAT_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/segformer_pths_16x/VermontLC_Segformer_CFAT_run2.pth
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_ESRGAN_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/s2naip/segformer_pths_16x/VermontLC_Segformer_ESRGAN_run2.pth
/glade/derecho/scratch/lizhi

In [7]:
def input_pipeline_downstream(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):

    feature_description = {
        'hres': tf.io.FixedLenFeature([4*hres_size_4x*hres_size_4x], dtype=tf.int64),
        'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
    }
    
    @tf.function
    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)

        hres = feature_dict['hres']
        hres = tf.reshape(hres, [4, hres_size_4x, hres_size_4x])
        hres = tf.cast(hres, tf.float32)
        hres = hres/255

        label = feature_dict['label']
        label = tf.reshape(label, [label_size, label_size, 1])

        return hres, label[..., 0]
        
    @tf.function
    def _augment_function(hres_img, label):
        # Transpose to [H, W, C]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
        if tf.rank(label) == 2:
            label = tf.expand_dims(label, axis=-1)
    
        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
        hres_img = tf.image.rot90(hres_img, k=k)
        label = tf.image.rot90(label, k=k)

        # ---- Random horizontal flip ----
        do_flip_lr = tf.random.uniform([]) > 0.5
        hres_img = tf.cond(do_flip_lr,
                           lambda: tf.image.flip_left_right(hres_img),
                           lambda: hres_img)
        label = tf.cond(do_flip_lr,
                        lambda: tf.image.flip_left_right(label),
                        lambda: label)
    
        # ---- Random vertical flip ----
        do_flip_ud = tf.random.uniform([]) > 0.5
        hres_img = tf.cond(do_flip_ud,
                           lambda: tf.image.flip_up_down(hres_img),
                           lambda: hres_img)
        label = tf.cond(do_flip_ud,
                        lambda: tf.image.flip_up_down(label),
                        lambda: label)
    
        # Transpose back to [C, H, W]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
        label = tf.squeeze(label, axis=-1)
    
        return hres_img, label

    dataset = tf.data.TFRecordDataset(filename)
    dataset = dataset.skip(skip)
    if take:
        dataset = dataset.take(take)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)

    return batch

# filenames = '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/ChesapeakeRSC_CAMixer_x16.tfrecords'
# ds = input_pipeline_downstream(filenames, 10, 0, 200, is_shuffle=True, is_train=True, is_repeat=True)


# for lres_batch, hres_batch in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))
#         hres_img = hres_batch[i].numpy()

#         axes[0].imshow(lres_img[:, :, :3])
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img)
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         plt.show()

#     break

In [8]:
model_order = ["ATD", "RGT", "CFAT", "CAMixer", "SRNO", "BiDiff", "SED", "ESRGAN", "UPSR"]

order_map = {m: i for i, m in enumerate(model_order)}

def extract_model_name(path):
    # ChesapeakeRSC_<MODEL>_x16.tfrecords
    return path.split('_')[-2]

# Sort files by model order
output_tfrecords_files = sorted(
    output_tfrecords_files,
    key=lambda x: order_map.get(extract_model_name(x), float('inf'))
)

for f in output_tfrecords_files:
    print(f)

/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_ATD_x16.tfrecords
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_RGT_x16.tfrecords
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_CFAT_x16.tfrecords
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_CAMixer_x16.tfrecords
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_SRNO_x16.tfrecords
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_BiDiff_x16.tfrecords
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_SED_x16.tfrecords
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_ESRGAN_x16.tfrecords
/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets/VermontLC_UPSR_x16.tfrecords


In [ ]:
# file_folder = '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets'
# output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]

# storage: {row_name: {model_name: value}}
mf1_results = {}
acc_results = {}

def extract_model_name(filename):
    # e.g. ChesapeakeRSC_SRNO_x16.tfrecords → SRNO
    return filename.split('_')[-2]

def make_row_name(dataset_name, run_id):
    return f"{dataset_name}_run{run_id}"

for run_id in range(1,4):
    for output_tfrecords in output_tfrecords_files:
        filename = os.path.basename(output_tfrecords)
        model_name = extract_model_name(filename)
        print(model_name)
        row_name = make_row_name(dataset_name, run_id)
        print(row_name)
        
        dirname = '/glade/derecho/scratch/lizhili/s2naip/segformer_pths_16x/'  # one level up (/content/drive/MyDrive/GeoSR)
        filename = os.path.basename(output_tfrecords)                 # e.g. M2L8_River_SRNO_x4.tfrecords
        base = filename.replace(f"{dataset_name}_", "").replace("_x16.tfrecords", "")
        segformer_save_path =  os.path.join(dirname, f"{dataset_name}_SegFormer_{base}_run{run_id}.pth")
        print(output_tfrecords)
        print('Load: ', segformer_save_path)
    
        print('Begin Segformer Training')
        device = 'cuda'
        model_naip = SegFormerSegmentor(num_classes=class_num, num_channels=4).to(device)
        model_naip.load_state_dict(torch.load(segformer_save_path.replace(".pth", "_best.pth"), weights_only=True))
        model_naip.eval()
        print('load successful!')
    
        print('Begin Testing')
        test_ds = input_pipeline_downstream(output_tfrecords, 4, num_training, num_test, is_shuffle=False, is_train=False, is_repeat=False)
    
        correct_pixels = 0
        total_pixels = 0
    
        if class_num == 2:
            TP = 0
            total_gt = 0
            total_pred = 0
        else:
            TP_per_class = [0] * class_num
            gt_per_class = [0] * class_num
            pred_per_class = [0] * class_num
    
        for images, labels in test_ds:
            images = torch.from_numpy(images.numpy().astype('float32')).to(device)
    
            with torch.no_grad():
                logits = model_naip(images)
                # print(logits.shape)
                logits = F.interpolate(logits, size=(label_size, label_size), mode='bilinear', align_corners=False)
                preds = torch.argmax(logits, dim=1).cpu().numpy()  # [B, H, W]
    
            labels = labels.numpy()  # [B, H, W]
            correct_pixels += np.sum(preds == labels)
            total_pixels += np.prod(labels.shape)
    
            if class_num == 2:
                TP += np.sum((preds == 1) & (labels == 1))
                total_gt += np.sum(labels == 1)
                total_pred += np.sum(preds == 1)
            else:
                for i in range(start_class, class_num):  # assuming class 0 is background or ignored
                    gt_mask = (labels == i)
                    pred_mask = (preds == i)
                    TP_per_class[i] += np.sum(gt_mask & pred_mask)
                    gt_per_class[i] += np.sum(gt_mask)
                    pred_per_class[i] += np.sum(pred_mask)
    
        # Accuracy
        accuracy = correct_pixels / total_pixels
        print("Accuracy:", round(accuracy, 4))
    
        # F1 Score
        if class_num == 2:
            mf1 = 2 * TP / (total_gt + total_pred) if (total_gt + total_pred) > 0 else 0
            print("F1 Score:", round(mf1, 4))
        else:
            f1_sum = 0
            valid_class_count = 0
            for i in range(start_class, class_num):
                denom = gt_per_class[i] + pred_per_class[i]
                f1 = 2 * TP_per_class[i] / denom if denom > 0 else 0
                print(f"Class {i} F1:", round(f1, 4))
                f1_sum += f1
                valid_class_count += 1 if denom > 0 else 0
            mf1 = f1_sum / valid_class_count
            print("Mean F1:", round(mf1, 4))
    
        # ----- store -----
        mf1_results.setdefault(row_name, {})[model_name] = mf1
        acc_results.setdefault(row_name, {})[model_name] = accuracy
    


In [12]:
import pandas as pd

mf1_df = pd.DataFrame.from_dict(mf1_results, orient='index').sort_index()
acc_df = pd.DataFrame.from_dict(acc_results, orient='index').sort_index()

# Add average row (mean over rows, i.e., runs/datasets)
mf1_df.loc["Average"] = mf1_df.mean(axis=0)
acc_df.loc["Average"] = acc_df.mean(axis=0)

mf1_df.to_excel(f"{dataset_name}_Segformer_mf1_UPSR.xlsx")
acc_df.to_excel(f"{dataset_name}_Segformer_accuracy_UPSR.xlsx")

print("Saved mf1.xlsx and accuracy.xlsx")

Saved mf1.xlsx and accuracy.xlsx


In [13]:
mf1_df

,UPSR
VermontLC_run1,0.382251
VermontLC_run2,0.379203
VermontLC_run3,0.375150
Average,0.378868
